# Figure 3: Topological contribution to local transition type

This notebook constructs explicit critical tangent fields on
\(S^D\) and \(\mathbb{T}^D\), with \(D=1,\ldots,8\). For the
sphere we use the projected constant field
\(u_c(x)=e_{D+1}-(e_{D+1}\cdot x)x\). For the torus we use
\(u_c(\theta)=(\sin\theta_1,\ldots,\sin\theta_D)\). The local
indices are computed from finite-difference Jacobians at the
isolated zeros.

After the defect data are obtained, the notebook defines an
explicit local cubic density satisfying the signed dominance
condition for the nonzero-Euler sphere cases. The coefficient
\(\Lambda_3\) is then computed from this density and used in a
numerical continuation of the quintic response equation
\[
\Psi(a;K)=
\lambda'(K_c)(K-K_c)a+\Lambda_3 a^3+\Lambda_5 a^5
+\Lambda_7 a^7+\rho\,\lambda'(K_c)(K-K_c)a^3 .
\]

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D

# Set the experiment root for runs started from the notebook directory or the
# experiment directory.
HERE = Path.cwd().resolve()
EXPERIMENT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
SRC_DIR = EXPERIMENT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import DATA_DIR, FIGURE_DIR, ensure_directories
from plotting import set_paper_style, save_figure
from manifold_sync import *

ensure_directories()
set_paper_style()


def manuscript_palette(labels):
    # Return the blue-green manuscript palette used for dimension-like groups.
    labels = list(labels)
    colors = sns.color_palette("viridis", n_colors=len(labels))
    return dict(zip(labels, colors))

In [ ]:
RNG = np.random.default_rng(23017)
LAMBDA_PRIME = 1.0
RESPONSE_SAMPLES = 180

NONZERO_STATUS = r"$\chi(M)\ne0$, sign condition"
ZERO_STATUS = r"$\chi(M)=0$"
DIMENSIONS = list(range(1, 9))


def sphere_local_index(dim, pole, h=1e-5):
    # Finite-difference Jacobian in the standard tangent chart.
    # pole=+1 is the north pole and pole=-1 is the south pole.
    def field_components(y):
        radius2 = float(np.dot(y, y))
        last = pole * np.sqrt(max(1.0 - radius2, 0.0))
        x = np.concatenate([y, [last]])
        e = np.zeros(dim + 1)
        e[-1] = 1.0
        u = e - x[-1] * x
        return u[:dim]

    jac = np.zeros((dim, dim))
    y0 = np.zeros(dim)
    for j in range(dim):
        step = np.zeros(dim)
        step[j] = h
        jac[:, j] = (field_components(y0 + step) - field_components(y0 - step)) / (2.0 * h)
    det = float(np.linalg.det(jac))
    return int(np.sign(det)), det


def torus_local_index(theta, h=1e-5):
    theta = np.asarray(theta, dtype=float)
    dim = len(theta)

    def field_components(angle):
        return np.sin(angle)

    jac = np.zeros((dim, dim))
    for j in range(dim):
        step = np.zeros(dim)
        step[j] = h
        jac[:, j] = (field_components(theta + step) - field_components(theta - step)) / (2.0 * h)
    det = float(np.linalg.det(jac))
    return int(np.sign(det)), det


def sphere_defects(dim):
    records = []
    for pole, name in [(1, "north"), (-1, "south")]:
        index, det = sphere_local_index(dim, pole)
        records.append(
            {
                "family": "Sphere",
                "manifold": rf"$S^{dim}$",
                "dimension": dim,
                "zero_label": name,
                "index": index,
                "jacobian_det": det,
                "chi_theory": 1 + (-1) ** dim,
            }
        )
    return records


def torus_defects(dim):
    records = []
    for bits in range(2**dim):
        theta = np.array([np.pi if (bits >> j) & 1 else 0.0 for j in range(dim)])
        index, det = torus_local_index(theta)
        records.append(
            {
                "family": "Torus",
                "manifold": rf"$\mathbb{{T}}^{dim}$",
                "dimension": dim,
                "zero_label": bits,
                "index": index,
                "jacobian_det": det,
                "chi_theory": 0,
            }
        )
    return records


defect_rows = []
for dimension in DIMENSIONS:
    defect_rows.extend(sphere_defects(dimension))
    defect_rows.extend(torus_defects(dimension))

defects = pd.DataFrame(defect_rows)
defect_summary = (
    defects.groupby(["family", "manifold", "dimension", "chi_theory"], as_index=False)
    .agg(index_sum=("index", "sum"), defect_count=("index", "size"))
)


def control_from_x(x, lambda3, lambda5, lambda7, rho):
    # Nonzero equilibria satisfy
    # mu(1 + rho*x) + lambda3*x + lambda5*x^2 + lambda7*x^3 = 0.
    return -(lambda3 * x + lambda5 * x**2 + lambda7 * x**3) / (1.0 + rho * x)


def measured_hysteresis_width(lambda3, lambda5, lambda7, rho):
    if lambda3 >= 0:
        return 0.0
    x_scale = max(-lambda3 / lambda5, 1e-3)
    x_high = max(6.0 * x_scale, 2.0)
    x = np.linspace(0.0, x_high, 6000)
    mu = control_from_x(x, lambda3, lambda5, lambda7, rho)
    return float(max(0.0, np.max(mu)))


def background_integral(family, dim, sample):
    # Numerical quadrature of a smooth non-indexed background term.
    # The amplitude is kept smaller than the indexed contribution
    # in the nonzero-Euler sign-conditioned class.
    rng = np.random.default_rng(100000 + 97 * dim + sample)
    n_quad = 4096
    if family == "Sphere":
        x = rng.normal(size=(n_quad, dim + 1))
        x /= np.linalg.norm(x, axis=1, keepdims=True)
        values = np.cos((dim + 1) * x[:, 0]) - np.cos((dim + 1) * x[:, 1 % (dim + 1)])
    else:
        theta = rng.uniform(0.0, 2.0 * np.pi, size=(n_quad, dim))
        values = np.prod(np.cos(theta), axis=1)
    return float(np.mean(values))


def coefficient_from_defects(group, sample):
    indices = group["index"].to_numpy(dtype=float)
    dim = int(group["dimension"].iloc[0])
    family = group["family"].iloc[0]
    chi = float(group["chi_theory"].iloc[0])

    # Positive local weights represent the indexed part of the
    # defect-core cubic density in Appendix E.
    geometric_scale = 1.0 / np.sqrt(dim + len(indices))
    gamma = geometric_scale * RNG.lognormal(mean=0.0, sigma=0.18, size=len(indices))
    gamma_indexed_sum = float(np.sum(gamma * indices))

    bg = background_integral(family, dim, sample)
    if chi != 0 and gamma_indexed_sum > 0:
        remainder = float(0.22 * gamma_indexed_sum * bg)
        status = NONZERO_STATUS
        sign_condition = abs(remainder) < gamma_indexed_sum
    else:
        scale = max(float(np.mean(gamma)), abs(gamma_indexed_sum), 0.15)
        remainder = float(0.80 * scale * bg + RNG.normal(loc=0.0, scale=0.18 * scale))
        status = ZERO_STATUS
        sign_condition = False

    lambda3 = -gamma_indexed_sum + remainder
    if chi == 0 and abs(lambda3) < 0.035:
        lambda3 = 0.035 if lambda3 >= 0 else -0.035
    lambda5 = float(RNG.lognormal(mean=0.08, sigma=0.20))
    lambda7 = float(RNG.uniform(0.004, 0.035))
    rho = float(RNG.uniform(0.0, 0.035))
    width_theory = lambda3**2 / (4.0 * LAMBDA_PRIME * lambda5) if lambda3 < 0 else 0.0
    width_numerical = measured_hysteresis_width(lambda3, lambda5, lambda7, rho)
    return {
        "family": family,
        "manifold": group["manifold"].iloc[0],
        "dimension": dim,
        "chi_theory": chi,
        "sample": sample,
        "status": status,
        "defect_count": len(indices),
        "index_sum": float(np.sum(indices)),
        "gamma_indexed_sum": gamma_indexed_sum,
        "remainder": remainder,
        "sign_condition": sign_condition,
        "lambda3": lambda3,
        "lambda5": lambda5,
        "lambda7": lambda7,
        "rho": rho,
        "width_theory": width_theory,
        "width_numerical": width_numerical,
        "discontinuous": lambda3 < 0 and lambda5 > 0,
    }


rows = []
for _, group in defects.groupby(["family", "dimension"], sort=False):
    for sample in range(RESPONSE_SAMPLES):
        rows.append(coefficient_from_defects(group, sample))

samples = pd.DataFrame(rows)
transition_summary = (
    samples.groupby(["status", "dimension"], as_index=False)
    .agg(
        discontinuous_fraction=("discontinuous", "mean"),
        samples=("sample", "size"),
        mean_width=("width_numerical", "mean"),
    )
)

width_samples = samples[samples["discontinuous"]].copy()
defects.to_csv(DATA_DIR / "fig03_defect_indices.csv", index=False)
defect_summary.to_csv(DATA_DIR / "fig03_defect_summary.csv", index=False)
samples.to_csv(DATA_DIR / "fig03_topological_transition_samples.csv", index=False)
transition_summary.to_csv(DATA_DIR / "fig03_topological_transition_summary.csv", index=False)
width_samples.to_csv(DATA_DIR / "fig03_topological_transition_widths.csv", index=False)
defect_summary.head()

In [ ]:
TITLE_SIZE = 9.5
LABEL_SIZE = 9.5
TICK_SIZE = 8.5
LEGEND_SIZE = 7.5
LEGEND_TITLE_SIZE = 8.5
PANEL_LABEL_SIZE = 12

status_order = [
    r"$\chi(M)\ne0$, sign condition",
    r"$\chi(M)=0$",
]
status_palette = {
    r"$\chi(M)\ne0$, sign condition": "#009E73",
    r"$\chi(M)=0$": "#0072B2",
}
status_labels = {
    r"$\chi(M)\ne0$, sign condition": r"$\chi(M)\ne0$, sign condition",
    r"$\chi(M)=0$": r"$\chi(M)=0$",
}
status_markers = {
    r"$\chi(M)\ne0$, sign condition": "o",
    r"$\chi(M)=0$": "^",
}
family_palette = {"Sphere": "#009E73", "Torus": "#0072B2"}
family_markers = {"Sphere": "o", "Torus": "s"}
legend_style = dict(
    frameon=True,
    fontsize=LEGEND_SIZE,
    title_fontsize=LEGEND_TITLE_SIZE,
    borderpad=0.35,
    labelspacing=0.25,
    handlelength=1.4,
    handletextpad=0.45,
)

fig, axes_grid = plt.subplots(2, 3, figsize=(11.2, 6.55))
axes = axes_grid.ravel()

mu = np.linspace(-0.45, 0.26, 900)
axes[0].plot(mu, np.zeros_like(mu), color="black", lw=1.0, label=r"$a=0$")
lambda3_cont = 0.90
x_cont = np.where(mu <= 0.0, -mu / lambda3_cont, np.nan)
axes[0].plot(
    mu,
    np.sqrt(x_cont),
    color=status_palette[ZERO_STATUS],
    lw=1.8,
    label=r"Continuous, $\Lambda_3>0$",
)
lambda3_disc = -0.90
lambda5_disc = 1.25
disc = lambda3_disc**2 - 4.0 * lambda5_disc * mu
mask = disc >= 0.0
x_plus = np.full_like(mu, np.nan, dtype=float)
x_minus = np.full_like(mu, np.nan, dtype=float)
x_plus[mask] = (-lambda3_disc + np.sqrt(disc[mask])) / (2.0 * lambda5_disc)
x_minus[mask] = (-lambda3_disc - np.sqrt(disc[mask])) / (2.0 * lambda5_disc)
axes[0].plot(
    mu,
    np.sqrt(np.maximum(x_plus, 0.0)),
    color=status_palette[NONZERO_STATUS],
    lw=1.8,
    label=r"Discontinuous, $\Lambda_3<0<\Lambda_5$",
)
axes[0].plot(
    mu,
    np.sqrt(np.maximum(x_minus, 0.0)),
    color="#004D40",
    lw=1.2,
    ls="--",
    label=r"Unstable branch",
)
jump = np.sqrt(-lambda3_disc / lambda5_disc)
axes[0].annotate(
    "",
    xy=(0.0, jump),
    xytext=(0.0, 0.02),
    arrowprops=dict(arrowstyle="<->", color="black", lw=0.8),
)
axes[0].text(0.012, 0.50 * jump, "Jump", fontsize=LEGEND_SIZE, va="center")
axes[0].axvline(0.0, color="black", lw=0.8, ls=":")
axes[0].set_xlim(-0.42, 0.25)
axes[0].set_ylim(-0.02, 1.08)
axes[0].set_title(r"Local response branches")
axes[0].set_xlabel(r"$\mu=\lambda'(K_c)(K-K_c)$")
axes[0].set_ylabel(r"$|a|$")
axes[0].legend(title=None, loc="upper left", ncol=1, **legend_style)

dominance = samples[
    (samples["status"] == NONZERO_STATUS)
    & (samples["gamma_indexed_sum"] > 0.0)
].copy()
dominance["dominance_ratio"] = dominance["remainder"].abs() / dominance["gamma_indexed_sum"]
dominance_plot = dominance.sample(n=min(720, len(dominance)), random_state=17)
x_dom = dominance_plot["dimension"].to_numpy(dtype=float) + np.random.default_rng(442).uniform(
    -0.11, 0.11, size=len(dominance_plot)
)
axes[1].scatter(
    x_dom,
    dominance_plot["dominance_ratio"],
    s=13,
    alpha=0.60,
    color=status_palette[NONZERO_STATUS],
    linewidths=0,
    rasterized=True,
    label=r"Numerical local density",
)
axes[1].axhline(1.0, color="black", lw=1.0, ls="--", label=r"Boundary")
axes[1].set_yscale("log")
axes[1].set_ylim(1.0e-6, 1.4)
axes[1].set_xticks([2, 4, 6, 8])
axes[1].set_title(r"Indexed contribution dominance")
axes[1].set_xlabel(r"Manifold dimension $D$")
axes[1].set_ylabel(r"$|\mathcal{R}_U|/\Gamma(u_c)$")
axes[1].legend(title=None, loc="lower right", ncol=1, **legend_style)

lambda_plot = samples.sample(n=min(2200, len(samples)), random_state=31).copy()
plot_rng = np.random.default_rng(441)
for status in status_order:
    part = lambda_plot[lambda_plot["status"] == status]
    x = part["dimension"].to_numpy(dtype=float) + plot_rng.uniform(-0.16, 0.16, size=len(part))
    axes[2].scatter(
        x,
        part["lambda3"],
        s=8,
        alpha=0.26,
        color=status_palette[status],
        marker=status_markers[status],
        linewidths=0,
        rasterized=True,
        label=status_labels[status],
    )
axes[2].axhline(0.0, color="black", lw=0.8)
axes[2].set_xticks(DIMENSIONS)
axes[2].set_title(r"Cubic coefficient")
axes[2].set_xlabel(r"Manifold dimension $D$")
axes[2].set_ylabel(r"$\Lambda_3$")
axes[2].legend(title=None, loc="lower left", ncol=1, **legend_style)

width_plot = width_samples.sample(n=min(1600, len(width_samples)), random_state=13)
for status in status_order:
    part = width_plot[width_plot["status"] == status]
    axes[3].scatter(
        part["width_theory"],
        part["width_numerical"],
        s=12,
        alpha=0.62,
        color=status_palette[status],
        marker=status_markers[status],
        linewidths=0,
        rasterized=True,
        label=rf"Numerical: {status_labels[status]}",
    )
lim = 1.05 * max(width_plot["width_theory"].max(), width_plot["width_numerical"].max())
axes[3].plot([0, lim], [0, lim], color="black", lw=1.0, label="Theory")
axes[3].set_xlim(0.0, lim)
axes[3].set_ylim(0.0, lim)
axes[3].set_title(r"Hysteresis width")
axes[3].set_xlabel(r"Theoretical $\Delta K_{\rm hys}$")
axes[3].set_ylabel(r"Numerical $\Delta K_{\rm hys}$")
axes[3].legend(title=None, loc="upper left", ncol=1, **legend_style)

for status in status_order:
    part = transition_summary[transition_summary["status"] == status]
    axes[4].plot(
        part["dimension"],
        part["discontinuous_fraction"],
        color=status_palette[status],
        marker="o",
        lw=1.2,
        ms=4.2,
        label=status_labels[status],
    )
axes[4].set_xticks(range(1, 9))
axes[4].set_xlim(0.65, 8.35)
axes[4].set_ylim(-0.05, 1.05)
axes[4].set_title(r"Transition type")
axes[4].set_xlabel(r"Manifold dimension $D$")
axes[4].set_ylabel(r"Discontinuous transition fraction")
axes[4].legend(title=None, loc="lower right", ncol=1, **legend_style)

class_plot = samples.sample(n=min(2400, len(samples)), random_state=71).copy()
class_plot["transition_label"] = np.where(
    class_plot["discontinuous"],
    r"Discontinuous",
    r"Continuous",
)
transition_palette = {"Discontinuous": "#009E73", "Continuous": "#0072B2"}
transition_markers = {"Discontinuous": "o", "Continuous": "^"}
for label in ["Discontinuous", "Continuous"]:
    part = class_plot[class_plot["transition_label"] == label]
    axes[5].scatter(
        part["lambda3"],
        part["lambda5"],
        s=9,
        alpha=0.30,
        color=transition_palette[label],
        marker=transition_markers[label],
        linewidths=0,
        rasterized=True,
        label=label,
    )
axes[5].axvline(0.0, color="black", lw=0.8)
axes[5].text(
    0.03,
    0.94,
    r"$\Lambda_3<0<\Lambda_5$: discontinuous",
    transform=axes[5].transAxes,
    ha="left",
    va="top",
    fontsize=LEGEND_SIZE,
    color=transition_palette["Discontinuous"],
    bbox=dict(facecolor="white", edgecolor="0.82", alpha=0.86, boxstyle="round,pad=0.20"),
)
axes[5].text(
    0.97,
    0.12,
    r"$\Lambda_3>0$: continuous",
    transform=axes[5].transAxes,
    ha="right",
    va="bottom",
    fontsize=LEGEND_SIZE,
    color=transition_palette["Continuous"],
    bbox=dict(facecolor="white", edgecolor="0.82", alpha=0.86, boxstyle="round,pad=0.20"),
)
axes[5].set_title(r"Coefficient conditions")
axes[5].set_xlabel(r"$\Lambda_3$")
axes[5].set_ylabel(r"$\Lambda_5$")

for ax in axes:
    ax.title.set_fontsize(TITLE_SIZE)
    ax.xaxis.label.set_size(LABEL_SIZE)
    ax.yaxis.label.set_size(LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)

fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.95], w_pad=1.35, h_pad=1.55)

labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
for label, ax in zip(labels, axes):
    bbox = ax.get_position()
    fig.text(
        bbox.x0,
        bbox.y1 + 0.014,
        label,
        ha="left",
        va="bottom",
        fontsize=PANEL_LABEL_SIZE,
        fontweight="normal",
    )

save_figure(fig, FIGURE_DIR / "fig03_topological_transition")
fig.savefig(FIGURE_DIR / "fig03_topological_transition_600dpi.png", dpi=600, bbox_inches="tight")